# ============================================================================
#  COLAB-READY: CAMOC Loss Pipeline for DBLP Dataset
#  Capacity-Aware Multi-Objective Contrastive Loss for Link Prediction
# ============================================================================

# %% [markdown]
# # CAMOC Loss: DBLP Dataset Evaluation
# This notebook adapts the Capacity-Aware Multi-Objective Contrastive (CAMOC)
# loss function for the standard PyTorch Geometric DBLP dataset.


In [1]:

# %% --- Cell 1: Install Dependencies ---
!pip install torch torch-geometric scikit-learn -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.8 MB/s eta 0:00:00


In [2]:
# %% --- Cell 2: Imports & Device ---
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (roc_auc_score, average_precision_score)
from torch_geometric.datasets import DBLP
from torch_geometric.nn import GCNConv, SAGEConv, GATConv, RGCNConv, HANConv
from torch_geometric.utils import negative_sampling
from torch.optim.lr_scheduler import CosineAnnealingLR
import time
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[*] CAMOC DBLP Pipeline — Device: {device}")

random.seed(42); np.random.seed(42); torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


[*] CAMOC DBLP Pipeline — Device: cpu


In [3]:

# %% --- Cell 3: Load DBLP Dataset ---
print("[*] Loading DBLP dataset...")
dataset = DBLP(root='./data/DBLP')
hdata = dataset[0]

print("\nDBLP Graph Stats:")
for nt in hdata.node_types:
    print(f"  Node type: {nt:10s} | Count: {hdata[nt].num_nodes} | Features: {hdata[nt].x.size() if 'x' in hdata[nt] else 'None'}")

for et in hdata.edge_types:
    print(f"  Edge type: {et} | Count: {hdata[et].edge_index.size(1)}")

# We will simulate Student-Advisor recommendation by predicting links between
# Author (Student) and Conference (Advisor).
# Since conferences don't have features in the default DBLP dataset, we'll create dummy features.
if 'x' not in hdata['conference']:
    # Create simple one-hot or random features for conferences
    hdata['conference'].x = torch.eye(hdata['conference'].num_nodes)

if 'x' not in hdata['author']:
    hdata['author'].x = torch.eye(hdata['author'].num_nodes)

if 'x' not in hdata['term']:
    hdata['term'].x = torch.eye(hdata['term'].num_nodes)

if 'x' not in hdata['paper']:
    hdata['paper'].x = torch.eye(hdata['paper'].num_nodes)

# Normalize all features to dimension 128 via initial linear projections later, or standardize dimensions.
node_types = list(hdata.node_types)
node_offsets = {}; offset = 0
for nt in node_types:
    node_offsets[nt] = offset
    offset += hdata[nt].num_nodes

total_num_nodes = offset

# Build homogeneous edge index for RGCN and negative sampling
all_src, all_dst, edge_types_list = [], [], []
for rel_idx, (s, r, d) in enumerate(hdata.edge_types):
    ei = hdata[s, r, d].edge_index
    if ei.size(1) > 0:
        all_src.append(ei[0] + node_offsets[s])
        all_dst.append(ei[1] + node_offsets[d])
        edge_types_list.extend([rel_idx] * ei.size(1))

homo_edge_index = torch.stack([torch.cat(all_src), torch.cat(all_dst)]).to(device)
homo_edge_type = torch.tensor(edge_types_list, dtype=torch.long).to(device)
type_labels = torch.cat([torch.full((hdata[nt].num_nodes,), idx, dtype=torch.long)
                          for idx, nt in enumerate(node_types)]).to(device)

print(f"\nHomogeneous Graph: {total_num_nodes} nodes, {homo_edge_index.size(1)} edges")



[*] Loading DBLP dataset...


Extracting data/DBLP/raw/DBLP_processed.zip
Processing...
Done!



DBLP Graph Stats:
  Node type: author     | Count: 4057 | Features: torch.Size([4057, 334])
  Node type: paper      | Count: 14328 | Features: torch.Size([14328, 4231])
  Node type: term       | Count: 7723 | Features: torch.Size([7723, 50])
  Node type: conference | Count: 20 | Features: None
  Edge type: ('author', 'to', 'paper') | Count: 19645
  Edge type: ('paper', 'to', 'author') | Count: 19645
  Edge type: ('paper', 'to', 'term') | Count: 85810
  Edge type: ('paper', 'to', 'conference') | Count: 14328
  Edge type: ('term', 'to', 'paper') | Count: 85810
  Edge type: ('conference', 'to', 'paper') | Count: 14328

Homogeneous Graph: 26128 nodes, 239566 edges


In [4]:
# %% --- Cell 4: CAMOC Loss Setup for DBLP ---
# In DBLP, let's treat 'author' as the student and 'conference' as the advisor.
student_type = 'author'
advisor_type = 'conference'

stu_start = node_offsets[student_type]
stu_end = stu_start + hdata[student_type].num_nodes
adv_start = node_offsets[advisor_type]
adv_end = adv_start + hdata[advisor_type].num_nodes
stu_range = (stu_start, stu_end)
adv_range = (adv_start, adv_end)

num_advisors = adv_end - adv_start
num_students = stu_end - stu_start

# Compute Semantic distance matrix for conferences (dummy or based on features)
# Using cosine distance on features
advisor_features = hdata[advisor_type].x
advisor_norm = F.normalize(advisor_features, dim=-1)
sem_sim_matrix = torch.mm(advisor_norm, advisor_norm.T)
sem_dist_matrix = (1.0 - sem_sim_matrix).clamp(0.0, 1.0)
sem_dist_matrix.fill_diagonal_(0.0)
sem_dist_matrix = sem_dist_matrix.to(device)

# Advisor Capacities and Loads
# Let's define a fixed capacity for each conference (e.g. 50 papers/authors)
advisor_caps = torch.full((num_advisors,), 50.0, device=device)

# Calculate actual load based on author->paper->conference connections
# Simplified load approximation
advisor_loads_raw = torch.zeros(num_advisors, device=device)
conf_edges = hdata['paper', 'to', 'conference'].edge_index
for i in range(conf_edges.size(1)):
    c_id = conf_edges[1, i].item()
    advisor_loads_raw[c_id] += 1.0

advisor_loads = (advisor_loads_raw / advisor_caps.clamp(min=1.0)).clamp(0.0, 2.0)

# Build a mapping of student (author) to their connected advisors (conferences) via papers
stu_pos_advisors = {}
ap_edges = hdata['author', 'to', 'paper'].edge_index
pc_edges = hdata['paper', 'to', 'conference'].edge_index

paper_to_conf = {}
for i in range(pc_edges.size(1)):
    p_id = pc_edges[0, i].item()
    c_id = pc_edges[1, i].item()
    if p_id not in paper_to_conf:
        paper_to_conf[p_id] = set()
    paper_to_conf[p_id].add(c_id)

for i in range(ap_edges.size(1)):
    a_id = ap_edges[0, i].item()
    p_id = ap_edges[1, i].item()
    if a_id not in stu_pos_advisors:
        stu_pos_advisors[a_id] = set()
    if p_id in paper_to_conf:
        stu_pos_advisors[a_id].update(paper_to_conf[p_id])

stu_with_advisors = [(s, list(advs)) for s, advs in stu_pos_advisors.items() if len(advs) > 0]
print(f"Author-Conference pairs via papers: {len(stu_with_advisors)}")



Author-Conference pairs via papers: 4057


In [5]:
# %% --- Cell 5: Models & Loss ---
class CAMOCLoss(nn.Module):
    def __init__(self, lambda1=0.15, lambda2=0.01, lambda3=0.3, alpha=0.15, m_base=0.02,
                 tau=0.1, eps=1e-8):
        super().__init__()
        self.lambda1 = lambda1
        self.lambda2 = lambda2
        self.lambda3 = lambda3
        self.alpha = alpha
        self.m_base = m_base
        self.tau = tau
        self.eps = eps
        self.bce = nn.BCEWithLogitsLoss()

    def adaptive_margin_bpr(self, pos_scores, neg_scores, sem_distances):
        margins = self.m_base + self.alpha * sem_distances
        diff = pos_scores - neg_scores - margins
        loss = -torch.log(torch.sigmoid(diff) + self.eps)
        return loss.mean()

    def cross_type_contrastive(self, student_embs, advisor_embs, pos_advisor_indices):
        s_norm = F.normalize(student_embs, dim=-1)
        a_norm = F.normalize(advisor_embs, dim=-1)
        logits = torch.mm(s_norm, a_norm.T) / self.tau
        labels = pos_advisor_indices.long()
        loss = F.cross_entropy(logits, labels)
        return loss

    def mwel_capacity_penalty(self, advisor_scores, load_ratios):
        overflow = torch.clamp(load_ratios - 1.0, min=0.0)
        penalty = overflow / (torch.exp(advisor_scores.detach()) + self.eps)
        return penalty.mean()

    def bce_auxiliary(self, pos_scores, neg_scores):
        scores = torch.cat([pos_scores, neg_scores])
        labels = torch.cat([torch.ones_like(pos_scores), torch.zeros_like(neg_scores)])
        return self.bce(scores, labels)

    def forward(self, pos_scores, neg_scores, sem_distances,
                student_embs, advisor_embs, pos_advisor_indices,
                advisor_match_scores, load_ratios, warmup_factor=1.0):
        L_rank = self.adaptive_margin_bpr(pos_scores, neg_scores, sem_distances)
        L_align = self.cross_type_contrastive(student_embs, advisor_embs, pos_advisor_indices)
        L_cap = self.mwel_capacity_penalty(advisor_match_scores, load_ratios)
        L_bce = self.bce_auxiliary(pos_scores, neg_scores)

        total = (L_rank + self.lambda1 * L_align + self.lambda2 * L_cap + self.lambda3 * L_bce) * warmup_factor
        return total, L_rank, L_align, L_cap

class StandardBCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.criterion = nn.BCEWithLogitsLoss()

    def forward(self, pos_scores, neg_scores):
        scores = torch.cat([pos_scores, neg_scores])
        labels = torch.cat([torch.ones_like(pos_scores), torch.zeros_like(neg_scores)])
        return self.criterion(scores, labels)

class UnifiedGNN(nn.Module):
    def __init__(self, in_channels_dict, hidden_dim, out_dim, gnn_type='GCN',
                 num_relations=10, heads=4):
        super().__init__()
        self.encoders = nn.ModuleDict({
            nt: nn.Linear(in_dim, hidden_dim)
            for nt, in_dim in in_channels_dict.items()
        })
        self.gnn_type = gnn_type

        if gnn_type == 'GCN':
            self.conv1 = GCNConv(hidden_dim, hidden_dim)
            self.conv2 = GCNConv(hidden_dim, out_dim)
        elif gnn_type == 'GraphSAGE':
            self.conv1 = SAGEConv(hidden_dim, hidden_dim)
            self.conv2 = SAGEConv(hidden_dim, out_dim)
        elif gnn_type == 'GAT':
            self.conv1 = GATConv(hidden_dim, hidden_dim // heads, heads=heads)
            self.conv2 = GATConv(hidden_dim, out_dim, heads=1)
        elif gnn_type == 'RGCN':
            self.conv1 = RGCNConv(hidden_dim, hidden_dim, num_relations)
            self.conv2 = RGCNConv(hidden_dim, out_dim, num_relations)

        self.bn = nn.BatchNorm1d(hidden_dim)

    def forward(self, x_dict, edge_index, edge_type=None):
        projected = []
        for nt in node_types:
            projected.append(self.encoders[nt](x_dict[nt].to(device)))
        x = torch.cat(projected, dim=0)

        if self.gnn_type == 'RGCN':
            x = F.relu(self.bn(self.conv1(x, edge_index, edge_type)))
            x = F.dropout(x, p=0.3, training=self.training)
            x = self.conv2(x, edge_index, edge_type)
        else:
            x = F.relu(self.bn(self.conv1(x, edge_index)))
            x = F.dropout(x, p=0.3, training=self.training)
            x = self.conv2(x, edge_index)
        return x

class HANModel(nn.Module):
    def __init__(self, in_channels_dict, hidden_dim, out_dim, metadata, heads=4):
        super().__init__()
        self.encoders = nn.ModuleDict({
            nt: nn.Linear(in_dim, hidden_dim)
            for nt, in_dim in in_channels_dict.items()
        })
        self.han1 = HANConv(hidden_dim, hidden_dim, metadata, heads=heads)
        self.han2 = HANConv(hidden_dim, out_dim, metadata, heads=1)

    def forward(self, x_dict, edge_index_dict):
        h_dict = {nt: F.relu(self.encoders[nt](x.to(device))) for nt, x in x_dict.items()}
        h_dict = self.han1(h_dict, edge_index_dict)
        h_dict = {nt: F.relu(h) for nt, h in h_dict.items()}
        h_dict = self.han2(h_dict, edge_index_dict)
        return h_dict

class LinkPredictor(nn.Module):
    def __init__(self, dim, hid=128):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(2*dim, hid), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hid, hid//2), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hid//2, 1)
        )
    def forward(self, zs, zd):
        return self.mlp(torch.cat([zs, zd], -1)).squeeze(-1)



In [6]:
# %% --- Cell 6: Training Utilities ---
def split_edges(ei, n_nodes, test_r=0.15, val_r=0.05):
    perm = torch.randperm(ei.size(1))
    nt, nv = int(ei.size(1)*test_r), int(ei.size(1)*val_r)
    return (ei[:, perm[nt+nv:]], ei[:, perm[nt:nt+nv]],
            negative_sampling(ei, n_nodes, num_neg_samples=nv),
            ei[:, perm[:nt]],
            negative_sampling(ei, n_nodes, num_neg_samples=nt))

def compute_mrr_and_hits(pos_scores, neg_scores, k=10):
    mrr_list, hits_list = [], []
    for pos_s in pos_scores:
        rank = 1 + torch.sum(neg_scores > pos_s).item()
        mrr_list.append(1.0 / rank)
        hits_list.append(1.0 if rank <= k else 0.0)
    return np.mean(mrr_list), np.mean(hits_list)

in_channels_dict = {nt: hdata[nt].x.size(1) for nt in node_types}
x_dict_dev = {nt: hdata[nt].x.to(device) for nt in node_types}
edge_index_dict_dev = {rel: hdata[rel].edge_index.to(device) for rel in hdata.edge_types}

train_ei, val_pos, val_neg, test_pos, test_neg = split_edges(homo_edge_index.cpu(), total_num_nodes)
train_ei = train_ei.to(device)
val_pos, val_neg = val_pos.to(device), val_neg.to(device)
test_pos, test_neg = test_pos.to(device), test_neg.to(device)

train_ei_set = set(zip(train_ei[0].cpu().tolist(), train_ei[1].cpu().tolist()))
train_edge_types = []
for i in range(homo_edge_index.size(1)):
    edge = (homo_edge_index[0, i].item(), homo_edge_index[1, i].item())
    if edge in train_ei_set:
        train_edge_types.append(homo_edge_type[i].item())
if len(train_edge_types) < train_ei.size(1):
    train_edge_types.extend([0] * (train_ei.size(1) - len(train_edge_types)))
train_edge_type_tensor = torch.tensor(train_edge_types[:train_ei.size(1)], dtype=torch.long).to(device)


In [7]:
# %% --- Cell 7: Training Loop ---
model_types = ['GCN', 'GraphSAGE', 'GAT', 'RGCN', 'HAN']
loss_types = ['BCE', 'CAMOC']

CAMOC_CONFIGS = {
    'GCN':       {'lambda1': 0.08, 'lambda2': 0.005, 'lambda3': 0.35, 'alpha': 0.20, 'm_base': 0.05, 'tau': 0.10, 'lr': 0.002, 'epochs': 250, 'warmup': 40, 'patience': 30},
    'GraphSAGE': {'lambda1': 0.06, 'lambda2': 0.004, 'lambda3': 0.30, 'alpha': 0.18, 'm_base': 0.04, 'tau': 0.08, 'lr': 0.002, 'epochs': 250, 'warmup': 40, 'patience': 30},
    'GAT':       {'lambda1': 0.05, 'lambda2': 0.003, 'lambda3': 0.35, 'alpha': 0.15, 'm_base': 0.03, 'tau': 0.12, 'lr': 0.001, 'epochs': 250, 'warmup': 50, 'patience': 35},
    'RGCN':      {'lambda1': 0.07, 'lambda2': 0.004, 'lambda3': 0.30, 'alpha': 0.18, 'm_base': 0.04, 'tau': 0.08, 'lr': 0.002, 'epochs': 250, 'warmup': 40, 'patience': 30},
    'HAN':       {'lambda1': 0.02, 'lambda2': 0.001, 'lambda3': 0.40, 'alpha': 0.10, 'm_base': 0.02, 'tau': 0.15, 'lr': 0.001, 'epochs': 250, 'warmup': 60, 'patience': 35},
}
BCE_CONFIG = {'lr': 0.005, 'epochs': 200, 'patience': 20}

all_results = {}

print("\n" + "="*72)
print("  EXPERIMENT: CAMOC vs BCE Loss (DBLP Dataset)")
print("="*72)

for loss_name in loss_types:
    print(f"\n{'='*72}")
    print(f"  Loss Function: {loss_name}")
    print(f"{'='*72}")

    for g_type in model_types:
        print(f"\n[*] Training [{g_type}] with [{loss_name}] loss...")
        start_time = time.time()

        if loss_name == 'CAMOC':
            cfg = CAMOC_CONFIGS[g_type]
            lr, max_epochs, patience, warmup_epochs = cfg['lr'], cfg['epochs'], cfg['patience'], cfg['warmup']
        else:
            lr, max_epochs, patience, warmup_epochs = BCE_CONFIG['lr'], BCE_CONFIG['epochs'], BCE_CONFIG['patience'], 0

        if g_type == 'HAN':
            model = HANModel(in_channels_dict, 64, 64, hdata.metadata()).to(device)
        else:
            model = UnifiedGNN(in_channels_dict, 64, 64, gnn_type=g_type,
                              num_relations=len(hdata.edge_types)).to(device)

        predictor = LinkPredictor(64).to(device)
        all_params = list(model.parameters()) + list(predictor.parameters())
        optimizer = torch.optim.Adam(all_params, lr=lr, weight_decay=5e-4)

        if loss_name == 'CAMOC':
            scheduler = CosineAnnealingLR(optimizer, T_max=max_epochs, eta_min=lr * 0.05)
            camoc_loss = CAMOCLoss(
                lambda1=cfg['lambda1'], lambda2=cfg['lambda2'], lambda3=cfg['lambda3'],
                alpha=cfg['alpha'], m_base=cfg['m_base'], tau=cfg['tau']
            ).to(device)
        else:
            scheduler = None
            bce_loss = StandardBCELoss().to(device)

        best_val_auc, patience_counter, best_state = 0, 0, None

        for epoch in range(1, max_epochs + 1):
            model.train(); predictor.train(); optimizer.zero_grad()

            if g_type == 'HAN':
                out_dict = model(x_dict_dev, edge_index_dict_dev)
                z = torch.cat([out_dict[nt] for nt in node_types], dim=0)
            elif g_type == 'RGCN':
                z = model(x_dict_dev, train_ei, train_edge_type_tensor)
            else:
                z = model(x_dict_dev, train_ei)

            pos_scores_all = predictor(z[train_ei[0]], z[train_ei[1]])
            neg_edges = negative_sampling(train_ei.cpu(), total_num_nodes, num_neg_samples=train_ei.size(1)).to(device)
            neg_scores_all = predictor(z[neg_edges[0]], z[neg_edges[1]])

            if loss_name == 'CAMOC':
                wf = float(epoch) / float(warmup_epochs) if epoch <= warmup_epochs else 1.0

                with torch.no_grad():
                    dst_sim = F.cosine_similarity(z[train_ei[1]], z[neg_edges[1]])
                    sem_dists_all = (1.0 - dst_sim).clamp(0.0, 1.0)
                margins_all = camoc_loss.m_base + camoc_loss.alpha * sem_dists_all
                bpr_diff = pos_scores_all - neg_scores_all - margins_all
                L_rank = -torch.log(torch.sigmoid(bpr_diff) + 1e-8).mean()

                n_cl = min(256, len(stu_with_advisors))
                if n_cl > 0:
                    cl_idx = random.sample(range(len(stu_with_advisors)), n_cl)
                    cl_s_list, cl_a_list = [], []
                    for idx in cl_idx:
                        s_local, advs = stu_with_advisors[idx]
                        cl_s_list.append(s_local)
                        cl_a_list.append(random.choice(advs))
                    cl_s = torch.tensor(cl_s_list, device=device)
                    cl_a = torch.tensor(cl_a_list, device=device)
                    s_emb = F.normalize(z[stu_range[0] + cl_s], dim=-1)
                    a_emb = F.normalize(z[adv_range[0]:adv_range[1]], dim=-1)
                    logits_cl = s_emb @ a_emb.T / camoc_loss.tau
                    L_align = F.cross_entropy(logits_cl, cl_a)
                else:
                    L_align = torch.tensor(0.0, device=device)

                overloaded = (advisor_loads > 1.0)
                if overloaded.any() and n_cl > 0:
                    ol_scores = predictor(
                        z[stu_range[0] + cl_s].mean(0, keepdim=True).expand(overloaded.sum(), -1),
                        z[adv_range[0]:adv_range[1]][overloaded]
                    )
                    overflow = advisor_loads[overloaded] - 1.0
                    clamped_scores = ol_scores.detach().clamp(-3.0, 3.0)
                    L_cap = (overflow / (torch.exp(clamped_scores) + 0.1)).mean().clamp(max=5.0)
                else:
                    L_cap = torch.tensor(0.0, device=device)

                L_bce_aux = camoc_loss.bce_auxiliary(pos_scores_all, neg_scores_all)
                loss = (L_rank + wf * camoc_loss.lambda1 * L_align + wf * camoc_loss.lambda2 * L_cap + camoc_loss.lambda3 * L_bce_aux)
            else:
                loss = bce_loss(pos_scores_all, neg_scores_all)

            loss.backward()
            if loss_name == 'CAMOC': torch.nn.utils.clip_grad_norm_(all_params, max_norm=1.0)
            optimizer.step()
            if scheduler is not None: scheduler.step()

            if epoch % 10 == 0 or epoch == 1:
                model.eval(); predictor.eval()
                with torch.no_grad():
                    if g_type == 'HAN':
                        out_dict = model(x_dict_dev, edge_index_dict_dev)
                        z = torch.cat([out_dict[nt] for nt in node_types], dim=0)
                    elif g_type == 'RGCN':
                        z = model(x_dict_dev, train_ei, train_edge_type_tensor)
                    else:
                        z = model(x_dict_dev, train_ei)
                    v_pos = predictor(z[val_pos[0]], z[val_pos[1]])
                    v_neg = predictor(z[val_neg[0]], z[val_neg[1]])
                    v_scores = torch.cat([v_pos, v_neg]).cpu().numpy()
                    v_labels = np.concatenate([np.ones(v_pos.size(0)), np.zeros(v_neg.size(0))])
                    val_auc = roc_auc_score(v_labels, 1 / (1 + np.exp(-v_scores)))

                if val_auc > best_val_auc:
                    best_val_auc = val_auc
                    best_state = {'model': {k: v.clone() for k, v in model.state_dict().items()}, 'predictor': {k: v.clone() for k, v in predictor.state_dict().items()}}
                    patience_counter = 0
                else:
                    patience_counter += 1
                    if patience_counter >= patience:
                        break

        if best_state is not None:
            model.load_state_dict(best_state['model'])
            predictor.load_state_dict(best_state['predictor'])

        model.eval(); predictor.eval()
        with torch.no_grad():
            if g_type == 'HAN':
                out_dict = model(x_dict_dev, edge_index_dict_dev)
                z = torch.cat([out_dict[nt] for nt in node_types], dim=0)
            elif g_type == 'RGCN':
                z = model(x_dict_dev, train_ei, train_edge_type_tensor)
            else:
                z = model(x_dict_dev, train_ei)
            t_pos = predictor(z[test_pos[0]], z[test_pos[1]])
            t_neg = predictor(z[test_neg[0]], z[test_neg[1]])
            t_scores = torch.cat([t_pos, t_neg]).cpu().numpy()
            t_labels = np.concatenate([np.ones(t_pos.size(0)), np.zeros(t_neg.size(0))])
            test_auc = roc_auc_score(t_labels, 1 / (1 + np.exp(-t_scores)))
            test_ap = average_precision_score(t_labels, 1 / (1 + np.exp(-t_scores)))
            test_mrr, test_hits = compute_mrr_and_hits(t_pos, t_neg, k=10)

        elapsed = time.time() - start_time
        result_key = f"{loss_name}_{g_type}"
        all_results[result_key] = {'auc': test_auc, 'ap': test_ap, 'mrr': test_mrr, 'hits': test_hits, 'time': elapsed}
        print(f"   => TEST AUC: {test_auc:.4f} | AP: {test_ap:.4f} | MRR: {test_mrr:.4f} | Hits@10: {test_hits:.4f} | Time: {elapsed:.1f}s")




  EXPERIMENT: CAMOC vs BCE Loss (DBLP Dataset)

  Loss Function: BCE

[*] Training [GCN] with [BCE] loss...
   => TEST AUC: 0.9766 | AP: 0.9744 | MRR: 0.0630 | Hits@10: 0.1402 | Time: 1119.0s

[*] Training [GraphSAGE] with [BCE] loss...
   => TEST AUC: 0.9673 | AP: 0.9673 | MRR: 0.0629 | Hits@10: 0.1290 | Time: 1090.1s

[*] Training [GAT] with [BCE] loss...
   => TEST AUC: 0.9696 | AP: 0.9658 | MRR: 0.0233 | Hits@10: 0.0569 | Time: 1180.0s

[*] Training [RGCN] with [BCE] loss...
   => TEST AUC: 0.9709 | AP: 0.9691 | MRR: 0.0364 | Hits@10: 0.0857 | Time: 1113.6s

[*] Training [HAN] with [BCE] loss...
   => TEST AUC: 0.9490 | AP: 0.9378 | MRR: 0.0092 | Hits@10: 0.0158 | Time: 1096.5s

  Loss Function: CAMOC

[*] Training [GCN] with [CAMOC] loss...
   => TEST AUC: 0.9759 | AP: 0.9749 | MRR: 0.1106 | Hits@10: 0.1567 | Time: 1378.4s

[*] Training [GraphSAGE] with [CAMOC] loss...
   => TEST AUC: 0.9670 | AP: 0.9656 | MRR: 0.0311 | Hits@10: 0.0942 | Time: 1326.3s

[*] Training [GAT] with [CA

In [8]:
# %% --- Cell 8: Comparative Results Table ---
print("\n" + "="*80)
print("  COMPARATIVE RESULTS: CAMOC vs BCE Loss Functions (DBLP)")
print("="*80)
print(f"{'Model':12s} | {'Loss':6s} | {'AUC':8s} {'AP':8s} {'MRR':8s} {'Hits@10':8s} | {'Time':6s}")
print("-" * 80)
for g_type in model_types:
    for loss_name in loss_types:
        r = all_results[f"{loss_name}_{g_type}"]
        print(f"{g_type:12s} | {loss_name:6s} | {r['auc']:.4f}   {r['ap']:.4f}   {r['mrr']:.4f}   {r['hits']:.4f}   | {r['time']:.1f}s")
    print("-" * 80)



  COMPARATIVE RESULTS: CAMOC vs BCE Loss Functions (DBLP)
Model        | Loss   | AUC      AP       MRR      Hits@10  | Time  
--------------------------------------------------------------------------------
GCN          | BCE    | 0.9766   0.9744   0.0630   0.1402   | 1119.0s
GCN          | CAMOC  | 0.9759   0.9749   0.1106   0.1567   | 1378.4s
--------------------------------------------------------------------------------
GraphSAGE    | BCE    | 0.9673   0.9673   0.0629   0.1290   | 1090.1s
GraphSAGE    | CAMOC  | 0.9670   0.9656   0.0311   0.0942   | 1326.3s
--------------------------------------------------------------------------------
GAT          | BCE    | 0.9696   0.9658   0.0233   0.0569   | 1180.0s
GAT          | CAMOC  | 0.9571   0.9547   0.0156   0.0445   | 1467.9s
--------------------------------------------------------------------------------
RGCN         | BCE    | 0.9709   0.9691   0.0364   0.0857   | 1113.6s
RGCN         | CAMOC  | 0.9700   0.9684   0.0417   0.1369 